# 01 — Data Pipeline: Generate Contrastive Triplets

This notebook generates the training data for the HCCS scorer.

**What it produces:** `data/triplets.jsonl` — ~1,500–3,000 contrastive triplets
of the form `(query, positive_context, negative_context, hallucination_type)`.

**How it works:**
1. Load tasks from CodeHaluEval (HuggingFace: `yuchen814/CodeHalu`)
2. For each task, sample 6 random context subsets from the repo chunks
3. Generate code with each subset using DeepSeek-Coder (or API)
4. Execute each generated code against the task's test cases
5. Pair a PASSING subset with a FAILING subset → contrastive triplet
6. Save triplets incrementally to Drive (Colab-safe)

**Runtime estimate:** ~10 hours on T4 GPU (~2 CU)

---
Run cells top-to-bottom. All cells are **idempotent** (safe to re-run after disconnect).

## Setup

In [ ]:
# Install dependencies (run once per Colab session)
!pip install torch transformers datasets rank-bm25 tqdm --quiet

In [ ]:
# Mount Google Drive for persistent storage
import sys
sys.path.insert(0, '..')

from notebooks.utils import mount_drive, check_gpu, print_env_summary

my_drive = mount_drive()
DEVICE = check_gpu()
print_env_summary()

In [ ]:
import json
import random
from pathlib import Path
from tqdm import tqdm

from haluguard.chunker import chunk_repo
from haluguard.efl import execute_code
from haluguard.data_pipeline import (
    ContrastiveTriplet,
    generate_context_subsets,
    save_triplets,
    load_triplets,
    summarise_triplets,
)
from notebooks.utils import append_jsonl, count_jsonl, get_drive_path

# Output paths
DRIVE_DIR = get_drive_path('HaluGuard/data')
TRIPLETS_PATH = DRIVE_DIR / 'triplets.jsonl'
print(f'Triplets will be saved to: {TRIPLETS_PATH}')
print(f'Existing triplets: {count_jsonl(TRIPLETS_PATH)}')

## 1. Load CodeHaluEval Dataset

In [ ]:
from datasets import load_dataset

# ~8,883 samples across 699 tasks
ds = load_dataset('yuchen814/CodeHalu', split='train')
print(f'Loaded {len(ds)} samples')
print('Columns:', ds.column_names)
print('\nFirst sample keys:', list(ds[0].keys()))

In [ ]:
# Inspect one sample to understand the data structure
sample = ds[0]
for key, val in sample.items():
    display_val = str(val)[:200] + '...' if len(str(val)) > 200 else val
    print(f'{key}: {display_val}')
    print()

## 2. Load Code Generation LLM

In [ ]:
# TODO: Load DeepSeek-Coder-1.3B (fits easily on T4)
# Option A: Local model on T4
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = 'deepseek-ai/deepseek-coder-1.3b-instruct'

# TODO: Uncomment when ready to run
# gen_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# gen_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16,
#     device_map='auto',
# )
# gen_model.eval()
# print(f'Model loaded on {next(gen_model.parameters()).device}')

print('LLM cell — uncomment above when ready to generate')

In [ ]:
def build_prompt(query: str, context_chunks: list) -> str:
    """Assemble a prompt from context chunks and the task query."""
    context_str = '\n\n'.join(context_chunks) if context_chunks else '(no context provided)'
    return (
        '# Repository context:\n\n'
        + context_str
        + '\n\n# Task:\n# '
        + query
        + '\n\n# Write the implementation below:\n'
    )


def generate_code(prompt: str, max_new_tokens: int = 256) -> str:
    """Generate code from a prompt using DeepSeek-Coder.
    
    TODO: implement with loaded gen_model / gen_tokenizer above,
    or replace with an API call (Anthropic, OpenAI, etc.).
    """
    raise NotImplementedError('Implement generate_code with your chosen LLM')


# Smoke test with a dummy prompt (will raise NotImplementedError — expected)
try:
    generate_code('test prompt')
except NotImplementedError as e:
    print(f'Expected: {e}')

## 3. Generate Contrastive Triplets

In [ ]:
# TODO: Run the triplet generation loop
# This is the main data generation cell — expect ~10 hours on T4

N_TASKS = 500        # number of CodeHaluEval tasks to process
N_SUBSETS = 6        # context subsets to try per task
SEED = 42

already_done = count_jsonl(TRIPLETS_PATH)
print(f'Resuming from {already_done} existing triplets')

# TODO: implement the loop
# for i, task in enumerate(tqdm(ds.select(range(N_TASKS)))):
#     query = task['prompt']          # adjust key to match dataset schema
#     repo_files = task['context']    # adjust key to match dataset schema
#     test_code = task['test']        # adjust key to match dataset schema
#     task_id = task['task_id']
#
#     chunks = chunk_repo(repo_files)
#     subsets = generate_context_subsets(chunks, n_subsets=N_SUBSETS, seed=SEED + i)
#
#     outcomes = []  # list of (context_str, passed, hallucination_type)
#     for subset in subsets:
#         prompt = build_prompt(query, subset)
#         code = generate_code(prompt)
#         result = execute_code(code, test_code)
#         outcomes.append(('\n\n'.join(subset), result.passed, result.hallucination_type))
#
#     # Form triplets: pair each PASS with each FAIL
#     passes = [(ctx, ht) for ctx, passed, ht in outcomes if passed]
#     fails  = [(ctx, ht) for ctx, passed, ht in outcomes if not passed]
#     for pos_ctx, _ in passes:
#         for neg_ctx, ht in fails:
#             triplet = ContrastiveTriplet(
#                 query=query,
#                 positive_context=pos_ctx,
#                 negative_context=neg_ctx,
#                 hallucination_type=ht,
#                 task_id=task_id,
#             )
#             append_jsonl(triplet.__dict__, TRIPLETS_PATH)

print('Loop stub — uncomment and implement above')

## 4. Analyse Triplets

In [ ]:
# Load and summarise collected triplets
if TRIPLETS_PATH.exists():
    triplets = load_triplets(TRIPLETS_PATH)
    summary = summarise_triplets(triplets)
    print(f'Total triplets: {summary["total"]}')
    print('By hallucination type:')
    for t, count in summary['by_hallucination_type'].items():
        pct = 100 * count / max(summary['total'], 1)
        print(f'  {t:10s}: {count:5d} ({pct:.1f}%)')
    print(f'Unknown type:    {summary["unknown"]}')
else:
    print(f'No triplets file found at {TRIPLETS_PATH}')
    print('Run the generation loop in cell above first.')

In [ ]:
# Copy triplets from Drive to the local repo data/ directory (optional)
import shutil
local_data = Path('../data')
local_data.mkdir(exist_ok=True)

if TRIPLETS_PATH.exists():
    dest = local_data / 'triplets.jsonl'
    shutil.copy(TRIPLETS_PATH, dest)
    print(f'Copied to {dest}')
else:
    print('No triplets to copy yet.')